# Kanasu hero: cursor-tracking frames (free, Google Colab)

Makes the 40 frames for the hero's cursor tracking with LivePortrait on Colab's free GPU.
Same settings as `scripts/generate-gaze-frames.mjs` / `generate_gaze_frames_free.py`.

1. **Runtime → Change runtime type → T4 GPU → Save**
2. **Runtime → Run all**
3. When step 2 asks, upload `character-gaze-source.png` from the portfolio folder.
4. At the end `gaze-frames.zip` downloads. Unzip it into `public/character/gaze/`.

## 1. Install LivePortrait (about 2 minutes)

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))

!git clone --depth=1 https://github.com/PowerHouseMan/ComfyUI-AdvancedLivePortrait.git advanced_live_portrait
!pip install -q ultralytics dill tyro timm lmdb rich imageio-ffmpeg

## 2. Upload `character-gaze-source.png`

In [ ]:
from google.colab import files
uploaded = files.upload()
SOURCE = next(iter(uploaded))
print('Source:', SOURCE)

## 3. Make the 40 frames (about 2–5 minutes)

In [ ]:
import os, sys, types, shutil
import numpy as np, torch, safetensors.torch
from PIL import Image

# The node is written for ComfyUI; stub the two ComfyUI modules it imports.
ROOT = os.getcwd()
MODELS = os.path.join(ROOT, 'models')
for d in ('models/liveportrait', 'models/ultralytics', 'temp', 'outputs'):
    os.makedirs(d, exist_ok=True)
fp = types.ModuleType('folder_paths')
fp.models_dir = MODELS
fp.output_directory = os.path.join(ROOT, 'outputs')
fp.get_folder_paths = lambda m: [os.path.join(MODELS, m)]
fp.get_save_image_path = lambda f, d, *a, **k: (d, f, 0, '', f)
fp.get_temp_directory = lambda: os.path.join(ROOT, 'temp')
fp.add_model_folder_path = lambda *a, **k: None
sys.modules['folder_paths'] = fp

comfy = types.ModuleType('comfy'); cu = types.ModuleType('comfy.utils')
cu.load_torch_file = lambda p, *a, **k: safetensors.torch.load_file(str(p)) if str(p).endswith('.safetensors') else torch.load(str(p), map_location='cpu', weights_only=False)
class ProgressBar:
    def __init__(self, *a, **k): pass
    def update(self, *a, **k): pass
    def update_absolute(self, *a, **k): pass
cu.ProgressBar = ProgressBar
comfy.utils = cu
sys.modules['comfy'] = comfy; sys.modules['comfy.utils'] = cu

from advanced_live_portrait.nodes import ExpressionEditor
editor = ExpressionEditor()

img = Image.open(SOURCE).convert('RGB')
src = torch.from_numpy(np.array(img, dtype=np.float32) / 255.0).unsqueeze(0)

# Same grid and settings as the portfolio scripts.
COLS, ROWS = 7, 5
def look(gx, gy):
    return dict(rotate_yaw=round(gx * 16, 1), rotate_pitch=round(gy * 10, 1), pupil_x=round(gx * 11, 1), pupil_y=round(gy * 7, 1))
jobs = [(f'r{r}c{c}.webp', look((c / (COLS - 1)) * 2 - 1, (r / (ROWS - 1)) * 2 - 1)) for r in range(ROWS) for c in range(COLS)]
jobs += [
    ('blink.webp', dict(blink=-14)),
    ('smile.webp', dict(smile=1.0, eyebrow=2)),
    ('talkA.webp', dict(aaa=45, smile=0.25)),
    ('talkO.webp', dict(woo=12, aaa=15)),
    ('talkE.webp', dict(eee=12, smile=0.3)),
]

KEYS = ['rotate_pitch', 'rotate_yaw', 'rotate_roll', 'blink', 'eyebrow', 'wink', 'pupil_x', 'pupil_y', 'aaa', 'eee', 'woo', 'smile']
OUT = 'gaze'
os.makedirs(OUT, exist_ok=True)
for i, (name, p) in enumerate(jobs, 1):
    res = editor.run(**{k: float(p.get(k, 0)) for k in KEYS}, src_ratio=1.0, sample_ratio=1.0,
                     sample_parts='OnlyExpression', crop_factor=1.7, src_image=src)
    arr = (res['result'][0].squeeze(0).detach().cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
    out = Image.fromarray(arr)
    if out.width > 1280:
        out = out.resize((1280, round(out.height * 1280 / out.width)), Image.LANCZOS)
    out.save(os.path.join(OUT, name), 'WEBP', quality=80)
    print(f'{i}/{len(jobs)} {name}')
print('All frames made.')

## 4. Quick look, then download `gaze-frames.zip`

In [ ]:
from IPython.display import display
names = ['r2c0.webp', 'r2c3.webp', 'r2c6.webp', 'r0c3.webp', 'r4c3.webp', 'blink.webp', 'smile.webp', 'talkA.webp']
thumbs = [Image.open(os.path.join(OUT, n)).resize((320, 180)) for n in names]
sheet = Image.new('RGB', (320 * 4, 180 * 2))
for i, t in enumerate(thumbs):
    sheet.paste(t, ((i % 4) * 320, (i // 4) * 180))
display(sheet)

shutil.make_archive('gaze-frames', 'zip', OUT)
files.download('gaze-frames.zip')